In [ ]:
# Mount Google Drive (if using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except:
    IN_COLAB = False

In [ ]:
!pip install bitsandbytes accelerate

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re
from typing import Dict, Any, List

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_path = "/content/drive/MyDrive/Phi_3_5_mini_instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
      quantization_config=bnb_config,
  torch_dtype=torch.float16,
    local_files_only=True
)

print("✅ Model loaded locally from Drive")

device = "cuda" if torch.cuda.is_available() else "cpu"

print("✅ Model loaded successfully")
print(f"✅ Device: {device}")
print(f"✅ Tokenizer loaded: {tokenizer is not None}")
print(f"✅ Model loaded: {model is not None}")

In [ ]:
import json
import re
import torch

def generate_text(
    prompt,
    tokenizer,
    model,
    device,
    do_sample=False,
    temperature=0.0,
    max_new_tokens=128,
    seed=42
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_tokens = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature
        )

    gen_tokens = outputs[0][input_tokens:]
    text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    return text

In [ ]:
import pandas as pd
df  = pd.read_csv("/content/drive/MyDrive/hf_models/agent_project/employees.csv")
df.columns

In [ ]:
dataset_schema= {
    "columns": list(df.columns),
    "dftype":{  col:str(df[col].dtype) for col in df.columns}

}
dataset_schema

In [ ]:
def ask_llm(user_input,SYSTEM_PROMPT):
    decision_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_input}
    ]

    decision_prompt = tokenizer.apply_chat_template(
        decision_messages,
        tokenize=False,
        add_generation_prompt=True
    )
    output = generate_text(
    decision_prompt,
    tokenizer,
    model,
    device,
    do_sample=False,       # 🔴 Stage 3
    temperature=0.0
    )
    return output

In [ ]:
SYSTEM_PROMPT_CODE = ( f"""
You are a pandas code generator.

Your job is to write ONE single line of Python code that answers the user question using a pandas DataFrame named df.

dataset schema: {dataset_schema}

STRICT RULES:
- Use  ONLY existing column names of dataset_schema exactly as written
1. The DataFrame already exists and its name is: df
2. Use ONLY pandas operations on df.
3. DO NOT import anything.
4. DO NOT define functions.
5. DO NOT create variables except: result
6. The final answer MUST be stored in a variable named: result
7. Output MUST be a SINGLE line of Python code.
8. DO NOT explain anything.
9. DO NOT add comments.
10. DO NOT output markdown or formatting.

Output format example:
result = df["salary"].mean()

Return ONLY the code line.
""")

In [ ]:
userQ = "What is the average salary of employees?"
ask_llm(userQ,SYSTEM_PROMPT_CODE)

In [ ]:
result = df["salary"].mean()
result

In [ ]:
userQ = "who have largest salary of employees?"
ask_llm(userQ,SYSTEM_PROMPT_CODE)

In [ ]:
result = df.nlargest(1, 'salary')['name'].values[0]
result

In [ ]:
userQ = "what largest salary of employees?"
ask_llm(userQ,SYSTEM_PROMPT_CODE)

In [ ]:
result = df["salary"].max()
result

In [ ]:
userQ = "avg exp"
gencode_raw= ask_llm(userQ,SYSTEM_PROMPT_CODE)
gencode_raw

In [ ]:
import re

def extract_code(text):
    match = re.search(r"result\s*=.*", text)
    if match:
        return match.group(0)
    raise ValueError("No valid code found")
gencode_extract = extract_code(gencode_raw)
gencode_extract

In [ ]:
def execute_llm_code(code, df):

    safe_globals = {
        "__builtins__": {}
    }

    safe_locals = {
        "df": df
    }

    exec(code, safe_globals, safe_locals)

    if "result" not in safe_locals:
        raise ValueError("LLM did not return result")

    return safe_locals["result"]

In [ ]:
execute_llm_code(gencode_extract, df)

## AST sandbox

In [ ]:
allowed_function = {"mean","avg","count","max"}
import ast
def valiate_code(code):
  tree = ast.parse(code)
  for node in ast.walk(tree):
    if isinstance(node, ast.Import):
      raise ValueError("Importing not allowed")
    if isinstance(node, ast.ImportFrom):
      raise ValueError("Importing not allowed")
    if isinstance(node, ast.While):
      raise ValueError("Loop not allowed")
    if isinstance(node, ast.FunctionDef):
      raise ValueError("Funtions not allowed")

    if isinstance(node, ast.Call):
      if isinstance(node.func, ast.Attribute):
        if node.func.attr not in allowed_function:
          raise ValueError("The function is not in allowed function")
  return True



In [ ]:
valiate_code("import os")

In [ ]:
valiate_code("result = df[""salary""].mean()")

## full system

In [ ]:
def fullsystem(df,userQ,SYSTEM_PROMPT_CODE):
  # Chech authorized??

  gencode_raw= ask_llm(userQ,SYSTEM_PROMPT_CODE)
  gencode_extract = extract_code(gencode_raw)
  print(gencode_extract)
  print("---"*20)
  # layer
  output = valiate_code(gencode_extract)
  if output:
    result = execute_llm_code(gencode_extract, df)
    return result

In [ ]:

userQ ="what is avg salary"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE)

In [ ]:
userQ ="what largest salary of employees?"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE)

In [ ]:
userQ ="who have largest salary of employees?"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE)

In [ ]:
state = {"authorized": False}

AUTHORIZED_SYSTEM_PROMPT =(
f"""
You are a security guard for a pandas DataFrame analysis system.

Your job is to decide whether the user's request is AUTHORIZED or UNAUTHORIZED.

The system only allows SAFE pandas aggregation queries on a DataFrame named df.
dataset schema: {dataset_schema}

AUTHORIZED requests:
- Questions that can be answered using pandas aggregation operations only.
- Examples: mean, sum, count, min, max, median, groupby aggregations.
- The request must return aggregated information, not raw rows.

UNAUTHORIZED requests include:
- Asking to import libraries
- Asking to write Python code
- Asking to use loops (for / while)
- Asking to access files or system resources
- Asking to show raw data such as df, df.head(), df.tail(), df.sample()
- Asking to display rows, records, or the entire dataset
- Asking to modify the dataframe
- Asking to create functions
- Any request unrelated to data aggregation

Decision rules:
If the question can be answered with safe pandas aggregation only → authorized = true
Otherwise → authorized = false

Return ONLY valid JSON."""
"""Output format:
{"authorized": true}
or

{"authorized": false}

Do not add explanations.
Do not add extra text.
Return JSON only.
""")


In [ ]:
import re
import json

def extract_json(text):
    match = re.search(r"\{.*?\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON found in text")

    json_str = match.group(0)
    return json.loads(json_str)

In [ ]:
userQ= "get avg salary"
action_raw= ask_llm(userQ,AUTHORIZED_SYSTEM_PROMPT)
action_raw

In [ ]:
userQ= "get first five rows in data"
action_raw= ask_llm(userQ,AUTHORIZED_SYSTEM_PROMPT)
action_raw

In [ ]:
def fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT):
  # Chech authorized??
  action_raw= ask_llm(userQ,AUTHORIZED_SYSTEM_PROMPT)
  action_raw = extract_json(action_raw)
  if action_raw["authorized"]:
      print("authorized")
      gencode_raw= ask_llm(userQ,SYSTEM_PROMPT_CODE)
      gencode_extract = extract_code(gencode_raw)
      print(gencode_extract)
      print("---"*20)
      # layer
      output = valiate_code(gencode_extract)
      if output:
        result = execute_llm_code(gencode_extract, df)
        return result
  elif action_raw["authorized"] == False:
    return "Unauthorized"

In [ ]:
userQ = "avg salary"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT)

In [ ]:
userQ = "get five rows"
fullsystem(df,userQ,SYSTEM_PROMPT_CODE,AUTHORIZED_SYSTEM_PROMPT)